# Основы PyTorch

**Содержание**
1. Тензоры и сходство с NumPy
2. CPU, GPU и измерение времени
3. Производная и `autograd`
4. Батчи и загрузка данных
5. Память, `item()`, `detach()` и `no_grad()`
6. Оптимизаторы: SGD с моментумом, RMSprop и Adam

In [1]:
import time
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Устройство:", device)

PyTorch: 2.14.1+cpu
Устройство: cpu


## 1. Тензоры и сходство с NumPy

Основная структура данных — `torch.Tensor`: многомерный массив, похожий на `numpy.ndarray`. У него есть форма (`shape`), тип элементов (`dtype`), индексация и операции над массивами. API похож, но не идентичен.

PyTorch добавляет автоматическое дифференцирование и вычисления на ускорителях. Для обучения обычно используют вещественные тензоры, например `float32`.

`*` — поэлементное умножение, `@` — матричное.

In [2]:
a = torch.tensor([[1., 2.], [3., 4.]])
b = torch.ones((2, 2))
print("Форма:", a.shape, "Тип:", a.dtype, "Устройство:", a.device)
print("Поэлементно:\n", a * b)
print("Матрично:\n", a @ b)
print("Среднее:", a.mean().item())

array = np.array([1., 2., 3.], dtype=np.float32)
tensor = torch.from_numpy(array)
print("Из NumPy:", tensor)
# from_numpy разделяет память с исходным массивом.
# Независимая копия: torch.from_numpy(array).clone()

Форма: torch.Size([2, 2]) Тип: torch.float32 Устройство: cpu
Поэлементно:
 tensor([[1., 2.],
        [3., 4.]])
Матрично:
 tensor([[3., 3.],
        [7., 7.]])
Среднее: 2.5
Из NumPy: tensor([1., 2., 3.])


## 2. CPU и GPU

`torch.cuda.is_available()` проверяет доступность CUDA для текущей установки PyTorch. Это не универсальная проверка наличия любой видеокарты. Команда `!nvidia-smi` в отдельной ячейке показывает сведения о NVIDIA GPU, если утилита установлена.

`x.to(device)` возвращает тензор на выбранном устройстве: результат нужно присвоить переменной. Для совместных вычислений параметры модели и входные тензоры должны находиться на совместимых устройствах.

Большие матричные операции часто выгодно выполнять на GPU. Чтение файлов и подготовка данных обычно начинаются на CPU, но предобработка и метрики также могут выполняться на GPU. Перенос на CPU нужен, например, для передачи результата в NumPy или scikit-learn. Для маленьких задач накладные расходы GPU могут превышать выигрыш.

In [3]:
x = torch.rand(3, 2)
x = x.to(device)
print("После переноса:", x.device)
x_cpu = x.cpu()
print("Обратно на CPU:", x_cpu.device)

После переноса: cpu
Обратно на CPU: cpu


### Измерение времени: вычисления и перенос данных

CUDA запускает операции асинхронно. Без ожидания завершения обычный таймер может измерить только постановку работы в очередь. В примере есть прогрев и `torch.cuda.synchronize()` перед и после измерения.

Сравним CPU, GPU с уже перенесёнными матрицами и GPU с переносом входов на каждой итерации. Результат остаётся на GPU; перенос результата обратно здесь не измеряется. Это учебный замер, а не гарантия ускорения на любом оборудовании.

[Документация `synchronize`](https://docs.pytorch.org/docs/stable/generated/torch.cuda.synchronize.html).

In [4]:
def layer(a, b):
    return (a @ b) ** 2


def measure_ms(fn, target_device, repeats=3):
    for _ in range(2):
        _ = fn()  # Прогрев.
    if target_device.type == "cuda":
        torch.cuda.synchronize(target_device)
    start = time.perf_counter()
    for _ in range(repeats):
        _ = fn()
    if target_device.type == "cuda":
        torch.cuda.synchronize(target_device)
    return (time.perf_counter() - start) * 1000 / repeats


# При желании увеличь размер до 2000, как в исходном черновике.
size = 512
A, B = torch.rand(size, size), torch.rand(size, size)
print(f"CPU: {measure_ms(lambda: layer(A, B), torch.device('cpu')):.2f} мс")
if device.type == "cuda":
    A_dev, B_dev = A.to(device), B.to(device)
    print(f"GPU, только вычисления: {measure_ms(lambda: layer(A_dev, B_dev), device):.2f} мс")
    print(f"GPU, с переносом входов: {measure_ms(lambda: layer(A.to(device), B.to(device)), device):.2f} мс")
else:
    print("CUDA недоступна: сравнение с GPU пропущено.")

CPU: 4.40 мс
CUDA недоступна: сравнение с GPU пропущено.


## 3. Производная и автоматическое дифференцирование

Производная показывает, как меняется функция при небольшом изменении аргумента. Для нескольких параметров набор частных производных называется **градиентом**.

Если $f(x)=x^2+3x$, то $f'(x)=2x+3$. В точке $x=2$ производная равна $7$.

В PyTorch:

- `requires_grad=True` включает отслеживание операций с тензором;
- прямой проход строит вычислительный граф;
- `backward()` применяет цепное правило и вычисляет градиенты;
- `.grad` хранит накопленный градиент у исходного тензора-листа.

Градиенты поддерживаются для вещественных и комплексных тензоров, поэтому ниже пишем `2.0`, а не целое `2`.

[Основы autograd](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html).

In [5]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3 * x
y.backward()
print("y =", y.item())       # 10
print("dy/dx =", x.grad.item())  # 7
assert torch.isclose(x.grad, torch.tensor(7.0))

y = 10.0
dy/dx = 7.0


### 3.1. Цепное правило

Пусть $u=2x+1$, а $y=u^2$. Тогда

$$\frac{dy}{dx}=\frac{dy}{du}\frac{du}{dx}=2u\cdot2=4(2x+1).$$

При $x=3$ получим $28$. PyTorch проходит по графу от результата к исходному аргументу. Промежуточный `u` участвует в вычислениях, но его `.grad` по умолчанию не сохраняется; при необходимости используют `u.retain_grad()`.

In [6]:
x = torch.tensor(3.0, requires_grad=True)
u = 2 * x + 1
u.retain_grad()
y = u**2
y.backward()
print("dy/du:", u.grad.item())  # 14
print("dy/dx:", x.grad.item())  # 28
assert x.grad.item() == 28.0

dy/du: 14.0
dy/dx: 28.0


### 3.2. Градиент вектора

Для $L=\sum_i x_i^2$ имеем $\partial L/\partial x_i=2x_i$.

`backward()` без аргумента ожидает скалярный результат. Поэтому в примере берём сумму. Для векторного результата можно передать внешний градиент: `(x**2).backward(torch.ones_like(x))` — здесь получится тот же результат, что после суммы.

In [7]:
x = torch.tensor([1., 2., 3.], requires_grad=True)
loss = (x**2).sum()
loss.backward()
print("Градиент:", x.grad)  # [2, 4, 6]
assert torch.equal(x.grad, 2 * x.detach())

Градиент: tensor([2., 4., 6.])


### 3.3. Почему градиенты нужно сбрасывать

`backward()` **прибавляет** вычисленные значения к `.grad`. Это позволяет намеренно накапливать градиенты нескольких батчей. В обычном цикле обучения их сбрасывают перед новым шагом.

Для отдельного тензора можно присвоить `x.grad = None`, а для параметров модели — вызвать `optimizer.zero_grad(set_to_none=True)`.

Каждый проход ниже строит новый граф. Повторный `backward()` по тому же графу обычно невозможен без его сохранения: после обратного прохода промежуточные данные освобождаются.

In [8]:
x = torch.tensor(2.0, requires_grad=True)
(x**2).backward()
print("После первого прохода:", x.grad.item())  # 4
(x**2).backward()
print("Без сброса:", x.grad.item())             # 8
x.grad = None
(x**2).backward()
print("После сброса:", x.grad.item())            # 4

После первого прохода: 4.0
Без сброса: 8.0
После сброса: 4.0


### 3.4. Один шаг градиентного спуска вручную

Минимизируем $L(w)=(w-3)^2$. Минимум находится при $w=3$.

$$w_{t+1}=w_t-\eta\,\frac{dL}{dw_t},$$

где $\eta$ — скорость обучения (`lr`). `backward()` вычисляет градиент, но сам параметр не изменяет. Обновление делаем отдельно под `no_grad()`, чтобы не включать его в граф.

In [9]:
w = torch.tensor(0.0, requires_grad=True)
lr = 0.1
for step in range(5):
    w.grad = None
    loss = (w - 3)**2
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
    print(f"Шаг {step + 1}: loss до шага={loss.item():.4f}, w после шага={w.item():.4f}")

Шаг 1: loss до шага=9.0000, w после шага=0.6000
Шаг 2: loss до шага=5.7600, w после шага=1.0800
Шаг 3: loss до шага=3.6864, w после шага=1.4640
Шаг 4: loss до шага=2.3593, w после шага=1.7712
Шаг 5: loss до шага=1.5099, w после шага=2.0170


## 4. Батчи и загрузка данных

**Батч** — группа примеров для одного прохода модели. Это позволяет использовать параллельные вычисления и не держать весь набор данных на GPU. За **эпоху** модель проходит весь обучающий набор.

`TensorDataset` объединяет признаки и цели, а `DataLoader` выдаёт батчи. `shuffle=True` перемешивает обучающие примеры при каждом новом проходе по загрузчику.

Создадим данные для зависимости $y=3x+2$ с небольшим шумом. Форма `[batch_size, 1]` означает один признак на объект.

In [10]:
torch.manual_seed(42)
features = torch.linspace(-1, 1, 256).unsqueeze(1)
targets = 3 * features + 2 + 0.05 * torch.randn_like(features)
dataset = TensorDataset(features, targets)
loader = DataLoader(dataset, batch_size=32, shuffle=True)
xb, yb = next(iter(loader))
print("Признаки:", xb.shape, "Цели:", yb.shape)
print("Батчей за эпоху:", len(loader))
xb, yb = xb.to(device), yb.to(device)
print("Устройство батча:", xb.device)

Признаки: torch.Size([32, 1]) Цели: torch.Size([32, 1])
Батчей за эпоху: 8
Устройство батча: cpu


## 5. Память и управление графом

Сохранение тензоров с графом в списках может удерживать связанные объекты и увеличивать расход памяти. После обычного `backward()` сохранённые для него промежуточные данные освобождаются, но это не делает хранение всех результатов бесплатным. Особенно опасно накапливать графы до обратного прохода или без необходимости использовать `retain_graph=True`.

| Операция | Что возвращает | Когда применять |
|---|---|---|
| `loss.item()` | Число Python из тензора с одним элементом | История ошибки, вывод в лог |
| `tensor.detach()` | Тензор без связи с графом, на том же устройстве | Результат для дальнейшей обработки без градиентов |
| `tensor.detach().cpu()` | Отсоединённый тензор на CPU | Сохранение предсказаний без удержания GPU-памяти |
| `with torch.no_grad():` | Контекст без записи операций в граф | Предсказание и оценка модели |

`item()` не меняет исходный тензор и не «отключает ему градиенты». `detach()` разделяет память с исходным тензором; для независимой копии нужен `detach().clone()`. Даже отсоединённые результаты занимают память, если складывать их в бесконечный список.

In [11]:
history = []
w = torch.tensor(2.0, requires_grad=True)
for _ in range(3):
    loss = (w - 1)**2
    history.append(loss.item())  # Числа, а не тензоры с графом.

prediction = w * torch.tensor([1., 2., 3.])
saved_prediction = prediction.detach().cpu()
print("История:", history)
print("Сохранённый результат:", saved_prediction)
print("requires_grad:", saved_prediction.requires_grad)

with torch.no_grad():
    inference_prediction = w * torch.tensor([1., 2., 3.])
print("Граф при предсказании:", inference_prediction.requires_grad)

История: [1.0, 1.0, 1.0]
Сохранённый результат: tensor([2., 4., 6.])
requires_grad: False
Граф при предсказании: False


### `eval()` и `no_grad()` решают разные задачи

`model.eval()` переключает поведение слоёв вроде Dropout и BatchNorm на режим оценки. Градиенты этот метод не отключает. `torch.no_grad()` отключает запись графа, но не меняет режим слоёв. Для оценки модели обычно используют оба.

In [12]:
demo_model = nn.Linear(1, 1).to(device)
demo_model.eval()
with torch.no_grad():
    prediction = demo_model(features[:5].to(device))
print("Предсказания необученной модели:\n", prediction.cpu().numpy())

Предсказания необученной модели:
 [[-1.4814848]
 [-1.4758813]
 [-1.470278 ]
 [-1.4646746]
 [-1.4590712]]


## 6. Оптимизаторы градиентного спуска

Оптимизатор обновляет параметры модели по градиентам. Он не вычисляет функцию потерь и не вызывает `backward()` автоматически в обычном цикле обучения.

Общий порядок шага:

1. `optimizer.zero_grad(set_to_none=True)` — сбросить старые градиенты.
2. `prediction = model(xb)` — получить предсказание.
3. `loss = loss_fn(prediction, yb)` — вычислить ошибку.
4. `loss.backward()` — вычислить градиенты.
5. `optimizer.step()` — обновить параметры.

Во всех формулах ниже $g_t=\nabla_\theta L_t$, $\eta$ — `lr`. Формулы приведены без регуляризации и дополнительных режимов.

### 6.1. SGD с моментумом

Обычный SGD делает шаг $\theta_{t+1}=\theta_t-\eta g_t$. На мини-батчах направление шумит; в вытянутых областях поверхности ошибки шаги могут колебаться.

**Моментум** накапливает направление прошлых градиентов:

$$v_t=\mu v_{t-1}+g_t,\qquad \theta_{t+1}=\theta_t-\eta v_t.$$

Здесь $v_0=0$, а $\mu$ — `momentum`. При согласованных градиентах движение ускоряется, при чередующихся — колебания могут сглаживаться. Скорость обучения всё равно нужно подбирать.

```python
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
```

`momentum=0` отключает накопление направления. Числа в примерах — отправная точка, а не универсальные настройки.

### 6.2. RMSprop

Если масштабы градиентов разных параметров сильно отличаются, единый размер шага может быть неудобен. RMSprop ведёт экспоненциальное среднее квадратов градиентов и масштабирует обновление каждого параметра:

$$s_t=\alpha s_{t-1}+(1-\alpha)g_t^2,$$
$$\theta_{t+1}=\theta_t-\eta\frac{g_t}{\sqrt{s_t}+\varepsilon}.$$

Квадрат, корень и деление здесь поэлементные. При большом среднем квадрате градиента эффективный шаг уменьшается. `alpha` управляет сглаживанием, `eps` стабилизирует деление.

```python
optimizer = torch.optim.RMSprop(
    model.parameters(), lr=0.001, alpha=0.99, eps=1e-8
)
```

Это адаптация масштаба шага; она не отменяет подбор `lr`. 

### 6.3. Adam

Adam объединяет сглаживание направления и адаптацию масштаба шага. Он хранит среднее градиентов (первый момент) и среднее их квадратов (второй момент):

$$m_t=\beta_1m_{t-1}+(1-\beta_1)g_t,$$
$$v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2.$$

Нулевые начальные значения смещают оценки в первых шагах; Adam корректирует это смещение:

$$\hat m_t=\frac{m_t}{1-\beta_1^t},\quad
\hat v_t=\frac{v_t}{1-\beta_2^t},\quad
\theta_{t+1}=\theta_t-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\varepsilon}.$$

```python
optimizer = torch.optim.Adam(
    model.parameters(), lr=0.001, betas=(0.9, 0.999), eps=1e-8
)
```

`betas` задаёт сглаживание двух моментов. Adam удобен как начальный вариант для эксперимента с шумными градиентами разных масштабов; качество всё равно оценивают на валидации. 

### 6.4. Как различаются оптимизаторы

| Оптимизатор | Что запоминает | Для чего |
|---|---|---|
| SGD с моментумом | Накопленное направление | Сглаживать колебания и ускорять согласованное движение |
| RMSprop | Среднее квадратов градиентов | Адаптировать масштаб шага для каждого параметра |
| Adam | Среднее градиентов и их квадратов | Совместить сглаживание направления и адаптацию шага |

Сравнивают качество на валидации, время и память, подбирая `lr` для каждого метода.

